In [ ]:
# ============================================
# Parkinson's Disease vs Control EEG Classifier
# Full Pipeline: BIDS loading, Preprocessing, ICA, Feature Extraction, ML
# Dataset: ds002778 (BIDS format)
# ============================================

import os
import os.path as op
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mne
from mne.datasets import sample
from mne_bids import BIDSPath, read_raw_bids, get_entity_vals
from mne.preprocessing import ICA, create_eog_epochs
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

## Step 1: Define Feature Extraction Function

In [ ]:
def extract_band_power_epochs(epochs, bands=[(1, 4), (4, 8), (8, 13), (13, 30)]):
    psd = epochs.compute_psd()
    psd_data, freqs = psd.get_data(return_freqs=True)

    X = []
    for trial in psd_data:  # loop over epochs
        features = []
        for fmin, fmax in bands:
            idx = np.logical_and(freqs >= fmin, freqs <= fmax)
            band_power = trial[:, idx].mean(axis=1)  # average power per channel
            features.extend(band_power)
        X.append(features)
    return X

## Step 2: Preprocess and Extract for Subject

In [ ]:
def process_subject(subject, session, label, bids_root, duration=5.0, overlap=1.0):
    try:
        bids_path = BIDSPath(
            root=bids_root, subject=subject, session=session,
            task='rest', datatype='eeg', suffix='eeg'
        )
        raw = read_raw_bids(bids_path=bids_path, verbose=False)

        # Handle missing or mislabeled channels
        raw.set_channel_types({
            "EXG1": "eog", "EXG2": "eog", "EXG3": "eog", "EXG4": "eog",
            "EXG5": "eog", "EXG6": "eog", "EXG7": "eog", "EXG8": "eog"
        })
        raw.set_montage("standard_1020", on_missing="ignore")

        raw.load_data()
        raw.filter(1., 40.)
        raw.notch_filter(60.)
        raw.set_eeg_reference('average')

        # ICA
        ica = ICA(n_components=20, random_state=97)
        ica.fit(raw)
        try:
            eog_epochs = create_eog_epochs(raw)
            eog_inds, _ = ica.find_bads_eog(eog_epochs)
            ica.exclude = eog_inds
        except Exception:
            ica.exclude = []
        raw = ica.apply(raw)

        # Epoching
        epochs = mne.make_fixed_length_epochs(raw, duration=duration, overlap=overlap, preload=True)

        if len(epochs) == 0:
            print(f"No epochs for {subject}, session {session}")
            return [], []

        # Feature extraction
        X_subject = extract_band_power_epochs(epochs)
        y_subject = [label] * len(X_subject)
        return X_subject, y_subject

    except Exception as e:
        if "number of columns changed" in str(e):
            print(f"❌ Skipping corrupted TSV file for {subject}, session {session}: {e}")
        else:
            print(f"❌ Failed to process {subject}, session {session}: {e}")
        return [], []

## Step 3: Loop Over All Subjects

In [ ]:
dataset = "ds002778"
bids_root = op.join(op.dirname(sample.data_path()), dataset)
all_subjects = get_entity_vals(bids_root, entity_key='subject')

X_all, y_all = [], []

for subject in all_subjects:
    if subject.startswith('pd'):
        label = 1
        # sessions = ['on', 'off']
        sessions = ['off']
    elif subject.startswith('hc'):
        label = 0
        sessions = ['hc']
    else:
        continue

    for session in sessions:
        X_subj, y_subj = process_subject(subject, session, label, bids_root)
        if X_subj:
            X_all.extend(X_subj)
            y_all.extend(y_subj)
            print(f"✅ Processed {subject}, session {session}, {len(X_subj)} samples")
        else:
            print(f"⚠️ No usable data from {subject}, session {session}")


In [ ]:
# ----------------------------
# Data Summary Report
# ----------------------------
n_total = len(y_all)
n_pd = sum(y_all)
n_control = n_total - n_pd

print("\n📊 Data Summary:")
print(f"Total samples (epochs): {n_total}")
print(f"Parkinson's (PD) samples: {n_pd}")
print(f"Control (HC) samples: {n_control}")
print(f"Total subjects processed: {len(set(all_subjects))}")


## Step 4: Train ML Classifier

### Train Random Forest Classifier

In [ ]:
# ----------------------------
# Step 4.1 : Train Random Forest Classifier
# ----------------------------

if len(X_all) == 0:
    raise ValueError("No data available for classification!")

X_all = np.array(X_all)
y_all = np.array(y_all)

# Custom implementation of RandomOverSampler
from collections import Counter
from sklearn.utils import resample

def custom_random_oversample(X, y, random_state=42):
    np.random.seed(random_state)
    unique_classes, class_counts = np.unique(y, return_counts=True)
    max_count = class_counts.max()

    X_balanced, y_balanced = [], []

    for cls in unique_classes:
        X_cls = X[y == cls]
        y_cls = y[y == cls]
        if len(X_cls) < max_count:
            X_resampled = resample(X_cls, replace=True, n_samples=max_count, random_state=random_state)
            y_resampled = np.full(max_count, cls)
        else:
            X_resampled = X_cls
            y_resampled = y_cls

        X_balanced.append(X_resampled)
        y_balanced.append(y_resampled)

    return np.vstack(X_balanced), np.concatenate(y_balanced)

X_resampled, y_resampled = custom_random_oversample(X_all, y_all)

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, stratify=y_resampled, test_size=0.2, random_state=42
)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("\n🔍 Classification Report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Control', 'PD'], yticklabels=['Control', 'PD'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
# ----------------------------
# Train Random Forest Classifier again
# ----------------------------

if len(X_all) == 0:
    raise ValueError("No data available for classification!")

X_all = np.array(X_all)
y_all = np.array(y_all)

# Alternative oversampling method: Equal replicate to max count
from collections import Counter

unique, counts = np.unique(y_all, return_counts=True)
max_count = np.max(counts)

X_bal, y_bal = [], []
for label in unique:
    X_cls = X_all[y_all == label]
    reps = max_count // len(X_cls)
    extra = max_count % len(X_cls)
    X_rep = np.concatenate([X_cls] * reps + [X_cls[:extra]])
    y_rep = np.full(max_count, label)
    X_bal.append(X_rep)
    y_bal.append(y_rep)

X_bal = np.vstack(X_bal)
y_bal = np.concatenate(y_bal)

X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal, stratify=y_bal, test_size=0.2, random_state=42
)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("\n🔍 Classification Report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Control', 'PD'], yticklabels=['Control', 'PD'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

### Step 4.2 : Train Support Vector Machinen Classifier

In [ ]:
from sklearn.svm import SVC


if len(X_all) == 0:
    raise ValueError("No data available for classification!")

X_all = np.array(X_all)
y_all = np.array(y_all)

# Balance class by oversampling the minority class
from collections import Counter

unique, counts = np.unique(y_all, return_counts=True)
max_count = np.max(counts)

X_bal, y_bal = [], []
for label in unique:
    X_cls = X_all[y_all == label]
    reps = max_count // len(X_cls)
    extra = max_count % len(X_cls)
    X_rep = np.concatenate([X_cls] * reps + [X_cls[:extra]])
    y_rep = np.full(max_count, label)
    X_bal.append(X_rep)
    y_bal.append(y_rep)

X_bal = np.vstack(X_bal)
y_bal = np.concatenate(y_bal)

X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal, stratify=y_bal, test_size=0.2, random_state=42
)

# Change classifier to Support Vector Machine
clf = SVC(kernel='rbf', C=1, gamma='scale', random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("\n🔍 Classification Report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Control', 'PD'], yticklabels=['Control', 'PD'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## Step 4.3: Train Logistic Regression Classifier

In [ ]:
from sklearn.linear_model import LogisticRegression

if len(X_all) == 0:
    raise ValueError("No data available for classification!")

X_all = np.array(X_all)
y_all = np.array(y_all)

# Balance class by oversampling the minority class
from collections import Counter

unique, counts = np.unique(y_all, return_counts=True)
max_count = np.max(counts)

X_bal, y_bal = [], []
for label in unique:
    X_cls = X_all[y_all == label]
    reps = max_count // len(X_cls)
    extra = max_count % len(X_cls)
    X_rep = np.concatenate([X_cls] * reps + [X_cls[:extra]])
    y_rep = np.full(max_count, label)
    X_bal.append(X_rep)
    y_bal.append(y_rep)

X_bal = np.vstack(X_bal)
y_bal = np.concatenate(y_bal)

X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal, stratify=y_bal, test_size=0.2, random_state=42
)

# Change classifier to Logistic Regression
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("\n🔍 Classification Report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Control', 'PD'], yticklabels=['Control', 'PD'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()